# NB06_fPINN — Finite-memory realizations of the learned fractional damper for real-time CTC

**Purpose.** First notebook of the finite-memory study (NB06–NB07). In NB03/NB04 the CTC + I adds the learned memory torque `B̂·D^α̂ q` evaluated with the **full-memory L1 sum** at the control period `T_C = 2 ms`. Its cost and RAM grow linearly with the elapsed time, so it cannot run indefinitely on a real controller. This notebook asks how much tracking accuracy is lost when that sum is replaced by realizations of **fixed cost per step**, and what each one costs:

| Method | Idea | Cost per update (3 joints) | RAM |
|---|---|---|---|
| `L1 full` | reference of NB03/NB04 | grows with t (6·n flops, n = t/T_C) | grows with t |
| `none` | memory term switched off (fPINN without its memory branch) | 0 | 0 |
| `window T_w` | short-memory principle: L1 sum over the last `T_w` only | 6·N_w | N_w samples |
| `window T_w + tail` | the same plus an O(1) correction of the discarded tail (summation by parts: `b_Nw·q̃(t−T_w) − (b_Nw − b_n)·mean(q̃)` over the old samples, `q̃ = q − q(0)`) | 6·N_w + O(1) | N_w samples |
| `Oustaloup n [ω_b, ω_h]` | band-limited rational approximation of `s^α` with n first-order modes, Tustin-discretized at `T_C` and written as parallel modes (a state-space form) | ≈ 3·(4n + 2) | 3·n states |

The networks are **not retrained**: the notebook reads the `fPINN` (and integer `PINN`, as the no-memory reference) weights of NB02/NB04 with 175 trajectories, α = 0.5 seeds 0–9 and α ∈ {0.3, 0.7, 0.9} seeds 0–2. The plant is exactly that of NB03/NB04 (full-memory Caputo damper, `B = 0.05`, `s = 1`, 2 kHz Euler–Cromer, `TAU_MAX = 5 N m`), and the controller is the same CTC + I (`Kp = 150`, `Kd = 10`, `Ki/Kp = 10`); only the realization of the memory term changes. The **fractional oracle** (exact τ_u, α, B) is run with every realization too, to separate model error from approximation error.

**Release v1.1.0.** The window + tail realization (Python `MemWindow` and C `win_update`) reads the discarded sample `q̃_{n−N_w−1}` from the ring buffer before the new sample overwrites the same slot. The first run of this notebook had the opposite order, so its `+ tail` (mean) rows averaged the recent past instead of the discarded history; NB07 (E0) quantifies the difference, and every `+ tail` result of the article comes from NB07.

**Experiments.**

1. *Offline error of the memory torque* (no networks): relative RMS error of each realization against the L1 sum on the Adept references, for α = 0.3 … 0.9, one cycle and continuous operation.
2. *Closed loop*: Adept 1.6 s (4 s cycle) and 3.2 s (8 s cycle) at 0 kg, plus **continuous operation**: `N_CYCLES` back-to-back 4 s cycles (60 s), where the history of the L1 sum is longest and the low-frequency limits of the cheap realizations show up. All realizations of one network are simulated together (one batch, identical arithmetic per member).
3. *Timing*: µs per update of each realization in Python and in plain C99 (`NB06_fPINN_memkernels.c`, written and, if `gcc` is found, compiled and run here) against the elapsed operation time (1 s … 1 h). The C file has no dependencies and can be compiled on an embedded CPU (Raspberry Pi: `gcc -O2 -std=c99 NB06_fPINN_memkernels.c -o memk -lm && ./memk > NB06_fPINN_TimingC_rpi.csv`); the update functions use no dynamic memory, so they can be pasted into an S-function for the dSPACE DS1103.

**One switch.** `QUICK = True` is a fast check (~1–2 min: α = 0.5, one network, four realizations, a 1.6 s cycle and 3 continuous cycles; if no weight file is found it uses a stand-in model and says so). `QUICK = False` runs **the whole experiment set of the article** with nothing else to edit.

**Expected runtime (Ryzen 7 5850U, 16 GB): about 1 h, between 40 min and 1.5 h** (closed loop ≈ 40 min measured on the development machine, scaled ×1.5 for the reference laptop: one batch of 14 realizations takes ~75 s on the development machine for the three scenarios, 70 % of it in the 60 s continuous run, and there are 23 such batches plus 19 single-realization PINN batches; the rest takes a few minutes). The progress line prints the time left from the speed it measures. Every closed-loop run is appended to the CSV and skipped on a rerun, so an interruption loses at most the batch in progress. RAM < 2 GB.

**Put in the same folder:** this notebook, the `NB02_fPINN_*_e150.npz` files of NB02 and the folder `NB04_fPINN_weights` of NB04. Optional: `NB04_fPINN_ClosedLoop.csv` (the notebook then checks that its `L1 full` runs reproduce the NB04 RMSE).

**Outputs:** `NB06_fPINN_ClosedLoop.csv`, `NB06_fPINN_ErrorMemoria.csv`, `NB06_fPINN_Timing.csv`, `NB06_fPINN_TimingC.csv` (if gcc was available), `NB06_fPINN_Summary.csv`, `fig_nb06_*.png`.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/Anaconda: same fix as NB04
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[_v] = "1"              # one BLAS thread (as on a controller); must be set before NumPy loads: restart the kernel first
import time, shutil, subprocess, numpy as np, pandas as pd
from math import gamma as Gamma
from scipy.signal import fftconvolve
from scipy import stats
import matplotlib.pyplot as plt

# ---- settings ---------------------------------------------------------------------------------
QUICK        = True                # True: fast check (~1-2 min). False: the whole experiment set of the article (see header).
B_FRAC       = 0.05                # [N m s^α] per joint (as NB02-NB04)
ALPHA_MAIN, MAIN_SEEDS = 0.5, tuple(range(10))
ALPHA_SWEEP, SWEEP_SEEDS = (0.3, 0.7, 0.9), (0, 1, 2)
SIZE, EPOCHS_TAG = 175, 150
SINGLE_SCALES = (1.6, 3.2)         # one Adept cycle (4 s, 8 s), 0 kg
CONT_SCALE, N_CYCLES = 1.6, 15     # continuous operation: 15 x 4 s = 60 s
PAYLOAD      = 0.0                 # 1.6 s with 6 kg saturates every model (NB03), not used here
NB02_WEIGHTS = "."
NB04_WEIGHTS = "NB04_fPINN_weights"
NB04_CL_CSV  = "NB04_fPINN_ClosedLoop.csv"
FS = 2000; dt = 1.0/FS; T_C = 0.002; STEP_C = int(round(T_C/dt))
TAU_MAX = 5.0
# realizations used in closed loop: (label, kind, parameters)
WINDOWS      = (0.1, 0.5, 2.0)     # [s]
OUST_MODES   = (3, 5, 7, 9)        # band [1e-2, 1e3] rad/s
OUST_BANDS   = ((1e-3, 1e3), (1e-1, 1e3))   # extra bands at 7 modes
TIMING_T     = (1, 10, 60, 600, 3600)       # elapsed operation time for the timing table [s]
if QUICK:
    MAIN_SEEDS, ALPHA_SWEEP, SINGLE_SCALES, N_CYCLES = (0,), (), (1.6,), 3
    WINDOWS, OUST_MODES, OUST_BANDS, TIMING_T = (0.5,), (7,), (), (1, 10, 60)
SUFFIX = "_QUICK" if QUICK else ""
OUT_CL = f"NB06_fPINN_ClosedLoop{SUFFIX}.csv"

def realizations():
    R = [("none", "none", {}), ("L1 full", "l1", {})]
    for tw in WINDOWS:
        R.append((f"window {tw:g} s", "win", dict(Tw=tw, tail=None)))
        R.append((f"window {tw:g} s + tail", "win", dict(Tw=tw, tail="mean")))
    for n in OUST_MODES: R.append((f"Oustaloup {n} [1e-2, 1e3]", "oust", dict(n=n, wb=1e-2, wh=1e3)))
    for wb, wh in OUST_BANDS: R.append((f"Oustaloup 7 [{wb:.0e}, {wh:.0e}]".replace("e-0", "e-").replace("e+0", "e"), "oust", dict(n=7, wb=wb, wh=wh)))
    if QUICK: R = [r for r in R if r[0] in ("none", "L1 full", "window 0.5 s + tail", "Oustaloup 7 [1e-2, 1e3]")]
    return R
REALIZ = realizations()
print("QUICK" if QUICK else "FULL", f"| {len(REALIZ)} realizations in closed loop: {[r[0] for r in REALIZ]}")

## Robot model and plant (NB03/NB04, written for a batch of robots)

Every function takes arrays with a leading batch axis `(M, 3)`; with `M = 1` the arithmetic is that of NB03. The plant's fractional damper uses the **full-memory L1 sum at 2 kHz** for every member; to keep 60 s of history affordable it is evaluated in blocks (old samples by FFT once per block, recent ones directly), which is the same sum up to rounding (checked below).

In [ ]:
R_BASE, R_PLAT, L_A, L_B = 0.114, 0.035, 0.200, 0.460
THETA0 = np.array([0.070, 0.023, 1e-5, 0.006, 0.253, 1.12, 0.008, 0.006, 0.006, 1.0, 0.008, 0.006, 0.006, 1.0])
M_BASE = 6.0
UNMODELED = dict(b_v=0.05, f_c=0.04, eps=0.02, c_cor=0.06, k_pos=0.04, w_pos=3.0)
def gamma_payload(m): return (M_BASE + float(m)) / M_BASE

def fk_b(q):
    """Forward kinematics of NB03 for q (M, 3) -> p (M, 3)."""
    rd = R_BASE - R_PLAT; C = np.zeros((len(q), 3, 3))
    for i in range(3):
        a = 2*i*np.pi/3; r = rd + L_A*np.cos(q[:, i])
        C[:, i, 0] = r*np.cos(a); C[:, i, 1] = r*np.sin(a); C[:, i, 2] = L_A*np.sin(q[:, i])
    d1 = 2*(C[:, 1] - C[:, 0]); d2 = 2*(C[:, 2] - C[:, 0]); A1, B1, C1 = d1.T; A2, B2, C2 = d2.T
    D1 = 2*L_A*rd*(np.cos(q[:, 0]) - np.cos(q[:, 1])); D2 = 2*L_A*rd*(np.cos(q[:, 0]) - np.cos(q[:, 2]))
    den = A2*B1 - A1*B2
    L1 = (A1*D2 - A2*D1)/den; L2 = (A1*C2 - A2*C1)/den; L3 = (B2*D1 - B1*D2)/den; L4 = (B2*C1 - B1*C2)/den
    u = 1 + L2**2 + L4**2; c0 = C[:, 0]
    v = 2*(L3*L4 + L1*L2 - c0[:, 0]*L4 - c0[:, 1]*L2 - c0[:, 2])
    w = c0[:, 0]**2 + c0[:, 1]**2 + c0[:, 2]**2 + L3**2 + L1**2 - 2*c0[:, 0]*L3 - 2*c0[:, 1]*L1 - L_B**2
    z = (-v + np.sqrt(v**2 - 4*u*w)) / (2*u)
    return np.column_stack([L3 + L4*z, L1 + L2*z, z])

def ik_vec(P):
    x, y, z = P[:, 0], P[:, 1], P[:, 2]; q = np.zeros((len(P), 3)); rd = R_BASE - R_PLAT
    for i in range(3):
        al = 2*i*np.pi/3; u = -2*L_A*z; aux = x*np.cos(al) + y*np.sin(al)
        v = 2*L_A*(rd - aux); w = L_B**2 - (rd**2 - 2*rd*aux + x**2 + y**2 + z**2 + L_A**2)
        r = np.sqrt(u**2 + v**2 - w**2); den = u**2 + v**2
        s1 = (u*w + v*r)/den; c1 = (v*w - u*r)/den; t1 = np.where(c1 >= 0, np.arcsin(s1), np.pi - np.arcsin(s1))
        s2 = (u*w - v*r)/den; c2 = (v*w + u*r)/den; t2 = np.where(c2 >= 0, np.arcsin(s2), np.pi - np.arcsin(s2))
        q[:, i] = np.where(np.abs(t1) < np.abs(t2), t1, t2)
    return q

def regressor_b(q, a, p):
    """(M, 3, 14) regressor of NB01 for a batch."""
    x, y, z = p[:, 0], p[:, 1], p[:, 2]; s, c = np.sin(q), np.cos(q); Y = np.zeros((len(q), 3, 14))
    Y[:, :, 0] = a; Y[:, :, 4] = c
    for i, (cx, cy, cz, cs) in enumerate([(1, 2, 3, 5), (6, 7, 8, 9), (10, 11, 12, 13)]):
        Y[:, i, cx] = -x*s[:, i]; Y[:, i, cy] = -y*s[:, i]; Y[:, i, cz] = z*c[:, i]; Y[:, i, cs] = s[:, i]
    return Y

def tau_unmodeled_b(q, qd):
    u = UNMODELED
    return (u["b_v"]*qd + u["f_c"]*np.tanh(qd/u["eps"])
            + u["c_cor"]*(qd**2*np.cos(q) + qd[:, [1, 2, 0]]*qd[:, [2, 0, 1]]) + u["k_pos"]*np.sin(u["w_pos"]*q))

def plant_qdd_b(q, qd, tau, p, m, Dq):
    g = gamma_payload(m)
    return (tau - g*(regressor_b(q, np.zeros_like(q), p) @ THETA0) - tau_unmodeled_b(q, qd) - B_FRAC*Dq) / (g*THETA0[0])

def F_nominal_b(q, qd, qdd, m, p): return gamma_payload(m)*(regressor_b(q, qdd, p) @ THETA0)
def F_oracle_b(q, qd, qdd, m, p):  return F_nominal_b(q, qd, qdd, m, p) + tau_unmodeled_b(q, qd)

def l1_weights(alpha, n): k = np.arange(n, dtype=float); return (k + 1.0)**(1 - alpha) - k**(1.0 - alpha)
def caputo_l1(x, h, alpha):
    """Batch L1 Caputo derivative of x (N, c) from t = 0 (reference)."""
    N = len(x); d = np.vstack([np.zeros((1, x.shape[1])), np.diff(x, axis=0)])
    return fftconvolve(d, l1_weights(alpha, N)[:, None], axes=0)[:N] * h**(-alpha) / Gamma(2 - alpha)

class PlantL1:
    """Full-memory L1 Caputo derivative of C channels sampled every h from t = 0, evaluated in blocks of L samples:
    samples older than the block by one FFT convolution per block, the rest by a direct sum. Same sum as NB03's FracHistory."""
    def __init__(self, alpha, h, n_max, C, L=1000):
        self.b = l1_weights(alpha, n_max + L + 2); self.c = h**(-alpha) / Gamma(2 - alpha); self.L = L
        self.br = self.b[:L][::-1].copy(); self.dq = np.zeros((n_max + 2, C)); self.n = 0; self.q_last = None; self.far = np.zeros((L, C))
    def push(self, q):
        if self.q_last is None: self.q_last = q.copy(); return np.zeros_like(q)
        self.n += 1; n = self.n; self.dq[n] = q - self.q_last; self.q_last = q.copy()
        if (n - 1) % self.L == 0:                     # new block starting at n0 = n: contribution of dq_1 ... dq_{n0-1}
            self.n0 = n
            if n > 1: self.far = fftconvolve(self.dq[1:n], self.b[:n - 1 + self.L][:, None], axes=0)[n - 1:n - 1 + self.L]
        j = n - self.n0
        return self.c * (self.far[j] + self.br[self.L - j - 1:] @ self.dq[self.n0:n + 1])

## Realizations of the controller's memory term `D^α̂ q`, updated every `T_C` from the measured angles

All share `reset()` and `push(q) -> D^α̂ q(t_n)` (3-vector), with t = 0 at the start of the run, as in NB03/NB04.

In [ ]:
class MemNone:
    def __init__(self, alpha): pass
    def reset(self, n_max): pass
    def push(self, q): return np.zeros(3)

class MemL1:
    """Full memory, direct sum (what a controller would execute; = FracHistory of NB03, single-threaded)."""
    def __init__(self, alpha, h=T_C): self.alpha, self.h = alpha, h; self.c = h**(-alpha)/Gamma(2 - alpha)
    def reset(self, n_max):
        self.b = l1_weights(self.alpha, n_max + 1)[::-1].copy(); self.dq = np.zeros((n_max + 1, 3)); self.n = 0; self.q_last = None
    def push(self, q):
        if self.q_last is None: self.q_last = q.copy(); return np.zeros(3)
        self.n += 1; self.dq[self.n] = q - self.q_last; self.q_last = q.copy()
        return self.c * np.einsum("i,ij->j", self.b[-self.n:], self.dq[1:self.n + 1])   # einsum: no BLAS call, so no thread overhead

class MemWindow:
    """L1 sum over the last N_w = T_w/h increments (ring buffer), plus an O(1) correction of the discarded part
    (summation by parts, q̃ = q − q_0): tail='lead' adds b_Nw·q̃_{n−Nw}; tail='mean' also subtracts (b_Nw − b_{n−1})·mean(q̃_1 … q̃_{n−Nw−1})."""
    def __init__(self, alpha, Tw, tail=None, h=T_C):
        self.alpha, self.h, self.tail = alpha, h, tail; self.Nw = max(1, int(round(Tw/h))); self.c = h**(-alpha)/Gamma(2 - alpha)
        self.bw = l1_weights(alpha, self.Nw + 1)
    def reset(self, n_max):
        Nw = self.Nw; self.buf = np.zeros((2*Nw, 3)); self.qb = np.zeros((Nw + 1, 3)); self.n = 0; self.q0 = None; self.q_last = None
        self.ssum = np.zeros(3); self.bk = l1_weights(self.alpha, n_max + 2) if self.tail == "mean" else None
        self.brev = self.bw[:Nw][::-1].copy()
    def push(self, q):
        if self.q0 is None: self.q0 = q.copy(); self.q_last = q.copy(); self.qb[0] = 0.0; return np.zeros(3)
        self.n += 1; n, Nw = self.n, self.Nw; d = q - self.q_last; self.q_last = q.copy()
        i = (n - 1) % Nw; self.buf[i] = d; self.buf[i + Nw] = d          # contiguous view of the last Nw increments
        m = min(n, Nw); D = np.einsum("i,ij->j", self.brev[Nw - m:], self.buf[i + Nw - m + 1:i + Nw + 1])
        if self.tail and n > Nw:
            D = D + self.bw[Nw]*self.qb[(n - Nw) % (Nw + 1)]              # leading term b_Nw·q̃_{n−Nw}
            if self.tail == "mean" and n - Nw - 1 >= 1:
                self.ssum += self.qb[(n - Nw - 1) % (Nw + 1)]            # q̃_{n−Nw−1}: read before its slot is overwritten
                D = D - (self.bw[Nw] - self.bk[n - 1])*self.ssum/(n - Nw - 1)
        self.qb[n % (Nw + 1)] = q - self.q0                              # q̃_n (overwrites q̃_{n−Nw−1})
        return self.c * D

class MemOustaloup:
    """Oustaloup approximation of s^α on [ω_b, ω_h] with n = 2N+1 modes, K·(1 + Σ r_k/(s + p_k)),
    each mode Tustin-discretized at h; input q̃ = q − q_0 (Caputo: zero initial state)."""
    def __init__(self, alpha, n, wb, wh, h=T_C):
        N = (n - 1)//2; k = np.arange(-N, N + 1)
        self.p = wb*(wh/wb)**((k + N + (1 + alpha)/2)/(2*N + 1)); z = wb*(wh/wb)**((k + N + (1 - alpha)/2)/(2*N + 1)); self.K = wh**alpha
        self.r = np.array([np.prod(z - pk)/np.prod(np.delete(self.p, i) - pk) for i, pk in enumerate(self.p)])
        self.A = ((2 - self.p*h)/(2 + self.p*h))[:, None]; self.Bd = (h/(2 + self.p*h))[:, None]
    def reset(self, n_max): self.X = np.zeros((len(self.p), 3)); self.u_prev = np.zeros(3); self.q0 = None
    def push(self, q):
        if self.q0 is None: self.q0 = q.copy()
        u = q - self.q0; self.X = self.A*self.X + self.Bd*(u + self.u_prev); self.u_prev = u
        return self.K*(u + self.r @ self.X)

def make_mem(kind, alpha, **kw):
    return {"none": MemNone, "l1": MemL1, "win": MemWindow, "oust": MemOustaloup}[kind](alpha, **kw)

# ---- checks -------------------------------------------------------------------------------------
_t = np.linspace(0, 6, 3001); _q = np.column_stack([np.sin(_t), 0.3*np.cos(2*_t), _t/6])
def online(mem, x): mem.reset(len(x)); return np.array([mem.push(v) for v in x])
_ref = caputo_l1(_q, T_C, 0.5)
print("L1 direct vs batch: max diff %.1e" % np.abs(online(MemL1(0.5), _q) - _ref).max())
print("window covering the whole run vs L1: max diff %.1e" % np.abs(online(MemWindow(0.5, 10.0), _q) - _ref).max())
_pl = PlantL1(0.5, T_C, len(_q), 3, L=400); print("block-FFT plant L1 vs batch: max diff %.1e" % np.abs(np.array([_pl.push(v) for v in _q]) - _ref).max())
_w = np.logspace(-3, 3.5, 300); _o = MemOustaloup(0.5, 7, 1e-2, 1e3)
_H = _o.K*(1 + (_o.r[None, :]/(1j*_w[:, None] + _o.p[None, :])).sum(1))
_band = (_w > 1e-1) & (_w < 1e2)
print("Oustaloup 7 [1e-2, 1e3], α = 0.5, continuous-time: max |gain error| %.2f dB, max phase error %.1f deg on [0.1, 100] rad/s"
      % (np.abs(20*np.log10(np.abs(_H[_band])/_w[_band]**0.5)).max(), np.abs(np.degrees(np.angle(_H[_band])) - 45).max()))

## References: one Adept cycle and continuous operation (back-to-back cycles)

In [ ]:
P_READY = np.array([0.0, 0.0, 0.5])
WAYPOINTS = [P_READY, np.array([.10, .10, .55]), np.array([.10, -.10, .55]),
             np.array([-.10, -.10, .55]), np.array([-.10, .10, .55]), np.array([.10, .10, .55]), P_READY]
BASE_T = np.array([0.0, 0.25, 0.75, 1.25, 1.75, 2.25, 2.50]); H_P = 0.05

def adept_P(scale):
    nt = BASE_T*scale; n = int(round(nt[-1]*FS)) + 1; t = np.arange(n)*dt
    seg = np.clip(np.searchsorted(nt[1:], t, side="right"), 0, len(WAYPOINTS) - 2)
    tau = np.clip((t - nt[seg]) / (nt[seg+1] - nt[seg]), 0, 1); s = 6*tau**5 - 15*tau**4 + 10*tau**3
    W = np.array(WAYPOINTS); P = W[seg] + (W[seg+1] - W[seg])*s[:, None]; P[:, 2] -= H_P*np.sin(np.pi*tau)**2
    return P

def reference(scale, cycles=1):
    P1 = adept_P(scale); P = np.vstack([P1] + [P1[1:]]*(cycles - 1)); n = len(P)
    q = ik_vec(P); qd = np.gradient(q, dt, axis=0, edge_order=2); qdd = np.gradient(qd, dt, axis=0, edge_order=2)
    return dict(t=np.arange(n)*dt, P=P, q=q, qd=qd, qdd=qdd, N=n, n_cycle=len(P1) - 1)

SCEN = {f"Adept {sc:g} s": (sc, 1) for sc in SINGLE_SCALES}; SCEN[f"Continuous {N_CYCLES}x{CONT_SCALE:g} s"] = (CONT_SCALE, N_CYCLES)
REFS = {k: reference(*v) for k, v in SCEN.items()}
for k, R in REFS.items(): print(f"{k:22s}: {R['t'][-1]:5.1f} s, {R['N']} steps, peak |qd| = {np.abs(R['qd']).max():.2f} rad/s")

## Experiment 1: offline error of the memory torque on the references (no networks)

Relative RMS error of `D^α q_ref` of every realization against the L1 sum at `T_C`, for each α. This table does not depend on the identified models; the closed loop below tells how much of this error reaches the tracking.

In [ ]:
OFF_METHODS = ([("none", "none", {})]
               + [(f"window {tw:g} s" + {None: "", "lead": " + lead term", "mean": " + tail"}[tl], "win", dict(Tw=tw, tail=tl))
                  for tw in ((0.05, 0.1, 0.2, 0.5, 1.0, 2.0) if not QUICK else (0.5,)) for tl in (None, "lead", "mean")]
               + [(f"Oustaloup {n} [{wb:.0e}, 1e3]", "oust", dict(n=n, wb=wb, wh=1e3))
                  for n, wb in ((3, 1e-2), (5, 1e-2), (7, 1e-2), (9, 1e-2), (11, 1e-2), (7, 1e-3), (9, 1e-3), (7, 1e-1))]
               )
off_rows = []; t0 = time.time()
for alpha in (0.3, 0.5, 0.7, 0.9):
    for scen, R in REFS.items():
        qc = R["q"][::STEP_C]; ref = caputo_l1(qc, T_C, alpha); s = np.sqrt(np.mean(ref**2))
        for label, kind, kw in OFF_METHODS:
            e = online(make_mem(kind, alpha, **kw), qc) - ref
            off_rows.append({"alpha": alpha, "Scenario": scen, "Realization": label, "Rel. RMS error (%)": 100*np.sqrt(np.mean(e**2))/s,
                             "Memory torque error RMS (mN m)": 1e3*B_FRAC*np.sqrt(np.mean(e**2)), "Memory torque RMS (mN m)": 1e3*B_FRAC*s})
off = pd.DataFrame(off_rows); off.to_csv(f"NB06_fPINN_ErrorMemoria{SUFFIX}.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 200)
print(f"({time.time() - t0:.0f} s) relative RMS error of D^α q (%), continuous operation:")
print(off[off.Scenario.str.startswith("Continuous")].pivot(index="Realization", columns="alpha", values="Rel. RMS error (%)").round(2).to_string())

## Models: fPINN and PINN networks of NB02/NB04 (175 trajectories), and the fractional oracle

In [ ]:
def silu(z): return z / (1.0 + np.exp(-z))
class NumpyFF:
    """Forward pass of NB02/NB04 for a batch (kinds MLP, HYBRID, PINN, fHYBRID, fPINN)."""
    def __init__(self, path):
        d = np.load(path, allow_pickle=False); self.kind = str(d["kind"]); self.core = {"fPINN": "PINN", "fHYBRID": "HYBRID"}.get(self.kind, self.kind)
        self.W = [d[k].astype(float) for k in ("W1", "b1", "W2", "b2", "W3", "b3")]
        self.x_mean, self.x_std = d["x_mean"].astype(float), d["x_std"].astype(float)
        self.y_mean, self.y_std, self.r_std = d["y_mean"].astype(float), d["y_std"].astype(float), d["r_std"].astype(float)
        self.alpha, self.B = float(d["alpha"]), d["B"].astype(float)
    def F(self, q, qd, qdd, m, p):
        W1, b1, W2, b2, W3, b3 = self.W
        xs = (np.column_stack([q, qd, qdd, np.full(len(q), m), p]) - self.x_mean) / self.x_std
        out = silu(silu(xs @ W1.T + b1) @ W2.T + b2) @ W3.T + b3
        if self.core == "MLP":    return out*self.y_std + self.y_mean
        if self.core == "HYBRID": return gamma_payload(m)*(regressor_b(q, qdd, p) @ THETA0) + out*self.r_std
        return np.einsum("mij,mj->mi", regressor_b(q, qdd, p), out)

def weight_file(kind, alpha, seed):
    scn = "main" if alpha == ALPHA_MAIN else "sweep"
    for f in (os.path.join(NB04_WEIGHTS, f"NB04_fPINN_{scn}_{kind}_{SIZE}traj_seed{seed}_a{alpha}_e{EPOCHS_TAG}.npz"),
              os.path.join(NB02_WEIGHTS, f"NB02_fPINN_{kind}_{SIZE}traj_seed{seed}_a{alpha}_e{EPOCHS_TAG}.npz")):
        if os.path.exists(f): return f
    return None

def model_list():
    """(alpha, label, seed, F, alpha_hat, B_hat, realizations) for every batch."""
    out = []; stand_in = False
    for alpha, seeds in [(ALPHA_MAIN, MAIN_SEEDS)] + [(a, SWEEP_SEEDS) for a in ALPHA_SWEEP]:
        out.append((alpha, "oracle (frac)", 0, F_oracle_b, alpha, np.full(3, B_FRAC), REALIZ))
        for seed in seeds:
            for kind in ("fPINN", "PINN"):
                f = weight_file(kind, alpha, seed)
                if f is None:
                    print(f"missing weights: {kind} {SIZE} traj seed {seed} α {alpha}")
                    if QUICK and kind == "fPINN":   # stand-in so that QUICK runs without weights: oracle dynamics with the typical fPINN bias
                        out.append((alpha, "fPINN (stand-in)", seed, F_oracle_b, alpha - 0.01, np.full(3, 1.1*B_FRAC), REALIZ)); stand_in = True
                    continue
                n = NumpyFF(f)
                out.append((alpha, kind, seed, n.F, n.alpha, n.B, REALIZ if n.B.any() else [("none", "none", {})]))
                if n.B.any(): print(f"α {alpha} {kind} seed {seed}: α̂ = {n.alpha:.3f}, B̂ = {np.round(n.B, 4)}  ({f})")
    if stand_in: print("QUICK without weights: 'fPINN (stand-in)' = oracle dynamics with α̂ = α − 0.01, B̂ = 1.1·B (checks the pipeline only)")
    return out
MODELS = model_list()
print(len(MODELS), "batches")

## Batched closed-loop simulation (CTC + I of NB03 + memory term from each realization)

In [ ]:
KP_PD, KD_PD, KI_RATIO = 150.0, 10.0, 10.0
KP_EQ, KD_EQ = KP_PD/THETA0[0], KD_PD/THETA0[0]

def simulate_batch(F, a_hat, B_hat, realiz, R, alpha, m=PAYLOAD):
    """All realizations of one model on one reference. Returns one metrics dict per realization (None if unstable)."""
    M = len(realiz); N = R["N"]; n_c = N//STEP_C + 2
    mems = [make_mem(kind, a_hat, **kw) for _, kind, kw in realiz]
    for mm in mems: mm.reset(n_c)
    q = np.repeat(R["q"][:1], M, 0); qd = np.repeat(R["qd"][:1], M, 0); I = np.zeros((M, 3)); mem = np.zeros((M, 3))
    plant = PlantL1(alpha, dt, N, 3*M); bad = np.zeros(M, bool)
    E = np.zeros((N, M)); T2 = np.zeros(M); TV = np.zeros(M); Tpk = np.zeros(M); Tprev = None
    MEM = np.zeros((n_c, M, 3)); k_c = 0; nc = R["n_cycle"]
    for i in range(N):
        p = fk_b(q); E[i] = np.linalg.norm(R["P"][i] - p, axis=1); Dq = plant.push(q.reshape(-1)).reshape(M, 3)
        if i % STEP_C == 0:
            for j in range(M): mem[j] = B_hat*mems[j].push(q[j])
            MEM[k_c] = mem; k_c += 1
        e = R["q"][i] - q; ed = R["qd"][i] - qd; I += e*dt
        v = R["qdd"][i] + KD_EQ*ed + KP_EQ*e + KI_RATIO*KP_EQ*I
        tau = np.clip(F(q, qd, v, m, p) + mem, -TAU_MAX, TAU_MAX)
        T2 += (tau**2).sum(1); Tpk = np.maximum(Tpk, np.abs(tau).max(1))
        if Tprev is not None: TV += np.linalg.norm(tau - Tprev, axis=1)
        Tprev = tau
        qd = qd + plant_qdd_b(q, qd, tau, p, m, Dq)*dt; q = q + qd*dt
        new_bad = ~np.all(np.isfinite(q), 1) | (np.abs(np.nan_to_num(q, nan=9.0)).max(1) > 3)
        if new_bad.any():                     # freeze a diverged member on the reference so it cannot disturb the others
            bad |= new_bad; q[new_bad] = R["q"][i + 1 if i + 1 < N else i]; qd[new_bad] = 0.0; I[new_bad] = 0.0
    e = 1000*E; MEM = MEM[:k_c]
    i_l1 = next((j for j, r in enumerate(realiz) if r[1] == "l1"), None)
    out = []
    for j, (label, kind, kw) in enumerate(realiz):
        if bad[j]: out.append(None); continue
        d = {"Cartesian RMSE (mm)": np.sqrt(np.mean(e[:, j]**2)), "Max error (mm)": e[:, j].max(),
             "RMSE first cycle (mm)": np.sqrt(np.mean(e[:nc + 1, j]**2)), "RMSE last cycle (mm)": np.sqrt(np.mean(e[-nc - 1:, j]**2)),
             "RMS torque (N m)": np.sqrt(T2[j]/(3*N)), "TV (N m)": TV[j], "Peak |tau| (N m)": Tpk[j],
             "Memory torque RMS (mN m)": 1e3*np.sqrt(np.mean(MEM[:, j]**2))}
        if i_l1 is not None: d["Memory torque diff vs L1 RMS (mN m)"] = 1e3*np.sqrt(np.mean((MEM[:, j] - MEM[:, i_l1])**2))
        out.append(d)
    return out

# check: one batch member reproduces the scalar loop of NB03 (oracle, L1 full, first 1.5 s of the 1.6 s cycle)
def fk(q): return fk_b(q[None])[0]
def _scalar_nb03(R, alpha, n):
    q = R["q"][0].copy(); qd = R["qd"][0].copy(); I = np.zeros(3); mem = MemL1(alpha); mem.reset(n//STEP_C + 2); Hp = MemL1(alpha, dt); Hp.reset(n); mm = np.zeros(3); P = []
    for i in range(n):
        p = fk(q); P.append(p); Dq = Hp.push(q)
        if i % STEP_C == 0: mm = B_FRAC*mem.push(q)
        e = R["q"][i] - q; ed = R["qd"][i] - qd; I += e*dt; v = R["qdd"][i] + KD_EQ*ed + KP_EQ*e + KI_RATIO*KP_EQ*I
        tau = np.clip(F_oracle_b(q[None], qd[None], v[None], PAYLOAD, p[None])[0] + mm, -TAU_MAX, TAU_MAX)
        qd = qd + plant_qdd_b(q[None], qd[None], tau[None], p[None], PAYLOAD, Dq[None])[0]*dt; q = q + qd*dt
    return 1000*np.sqrt(np.mean(np.linalg.norm(R["P"][:n] - np.array(P), axis=1)**2))
_R = reference(1.6); _n = 3000; _Rs = {k: (v[:_n] if isinstance(v, np.ndarray) else v) for k, v in _R.items()}; _Rs["N"] = _n
_b = simulate_batch(F_oracle_b, 0.5, np.full(3, B_FRAC), [("L1 full", "l1", {}), ("none", "none", {})], _Rs, 0.5)
print("batch vs scalar loop (oracle, L1 full, 1.5 s): RMSE %.6f vs %.6f mm" % (_b[0]["Cartesian RMSE (mm)"], _scalar_nb03(_R, 0.5, _n)))

## Experiment 2: closed loop, every model × scenario (resumable)

In [ ]:
key = lambda d: (float(d["alpha true"]), d["Model"], int(d["Seed"]), d["Scenario"])
rows = pd.read_csv(OUT_CL).to_dict("records") if os.path.exists(OUT_CL) else []
done = {key(r) for r in rows}
jobs = [(Mo, sc) for Mo in MODELS for sc in REFS if key({"alpha true": Mo[0], "Model": Mo[1], "Seed": Mo[2], "Scenario": sc}) not in done]
steps = sum(REFS[sc]["N"]*(0.3 + 0.7*len(Mo[6])/len(REALIZ)) for Mo, sc in jobs)
print(f"{len(jobs)} batches to run ({len(done)} already in {OUT_CL})")
t_all = time.time(); s_done = 0
for k, ((alpha, label, seed, F, a_hat, B_hat, realiz), sc) in enumerate(jobs, 1):
    t0 = time.time(); res = simulate_batch(F, a_hat, B_hat, realiz, REFS[sc], alpha)
    for (rl, _, _), r in zip(realiz, res):
        rows.append({"alpha true": alpha, "Model": label, "Seed": seed, "Scenario": sc, "Realization": rl, "alpha hat": a_hat,
                     "B hat mean": float(np.mean(B_hat)), "Batch time (s)": time.time() - t0, "Unstable": r is None, **(r or {})})
    pd.DataFrame(rows).to_csv(OUT_CL, index=False)
    s_done += REFS[sc]["N"]*(0.3 + 0.7*len(realiz)/len(REALIZ)); el = time.time() - t_all
    l1 = next((r for (rl, _, _), r in zip(realiz, res) if rl in ("L1 full", "none")), None)
    print(f"[{k}/{len(jobs)}] α {alpha} | {label:16s} s{seed} | {sc:20s} | {len(realiz):2d} realiz. | "
          + (f"RMSE ({'L1' if len(realiz) > 1 else 'no memory'}) {1e3*l1['Cartesian RMSE (mm)']:.1f} µm" if l1 else "UNSTABLE")
          + f" | {time.time() - t0:.0f} s | elapsed {el/60:.1f} min, left ≈ {el/s_done*(steps - s_done)/60:.0f} min", flush=True)
cl = pd.DataFrame(rows)
print(f"closed loop: {(time.time() - t_all)/60:.1f} min | unstable: {int(cl['Unstable'].astype(bool).sum())} of {len(cl)}")

# consistency with NB04 (same plant, controller and weights): L1 full of NB06 vs CTC + I of NB04
if os.path.exists(NB04_CL_CSV) and len(cl):
    o = pd.read_csv(NB04_CL_CSV); o = o[(o["Scenario"].isin(["main", "sweep"]) if "Scenario" in o else True) & (o["Data (traj.)"] == SIZE) & (o["Payload (kg)"] == PAYLOAD)]
    chk = []
    for _, r in cl[(cl.Realization.isin(["L1 full"])) | ((cl.Model == "PINN") & (cl.Realization == "none"))].iterrows():
        if not r["Scenario"].startswith("Adept"): continue
        sc = float(r["Scenario"].split()[1]); g = o[(o["alpha true"] == r["alpha true"]) & (o.Model == r["Model"]) & (o.Seed == r["Seed"]) & (o["Speed scale"] == sc)]
        if len(g): chk.append((r["Model"], r["Seed"], sc, r["Cartesian RMSE (mm)"], g["Cartesian RMSE (mm)"].iloc[0]))
    if chk:
        c = pd.DataFrame(chk, columns=["Model", "Seed", "Scale", "NB06", "NB04"]); c["rel diff"] = (c.NB06 - c.NB04).abs()/c.NB04
        print(f"NB06 L1 full vs NB04 ({len(c)} runs): max relative difference {c['rel diff'].max():.1e}  (should be < 1e-6)")

## Summary: tracking loss of each realization against the L1 sum, and share of the fPINN gain over the PINN that it keeps

For each α and scenario: mean Cartesian RMSE over seeds, paired difference with `L1 full` (µm and %, paired t-test over seeds), and **gain kept** = (RMSE_PINN − RMSE_realization) / (RMSE_PINN − RMSE_L1) over the same seeds (100 % = as good as full memory, 0 % = no better than the integer PINN).

In [ ]:
ok = cl[~cl["Unstable"].astype(bool)].copy(); summ = []
for (alpha, sc), g in ok.groupby(["alpha true", "Scenario"]):
    pinn = g[(g.Model == "PINN")].set_index("Seed")["Cartesian RMSE (mm)"]
    for model in [m for m in g.Model.unique() if m != "PINN"]:
        gm = g[g.Model == model]; l1 = gm[gm.Realization == "L1 full"].set_index("Seed")["Cartesian RMSE (mm)"]
        for rl in [r[0] for r in REALIZ]:
            x = gm[gm.Realization == rl].set_index("Seed")["Cartesian RMSE (mm)"]
            if not len(x): continue
            s = x.index.intersection(l1.index); d = 1e3*(x[s] - l1[s])
            row = {"alpha true": alpha, "Scenario": sc, "Model": model, "Realization": rl, "n seeds": len(x),
                   "RMSE mean (µm)": 1e3*x.mean(), "RMSE std (µm)": 1e3*x.std(), "Δ vs L1 (µm)": d.mean(), "Δ vs L1 (%)": 100*(d/(1e3*l1[s])).mean(),
                   "p (paired t, vs L1)": stats.ttest_rel(x[s], l1[s]).pvalue if len(s) >= 3 and np.any(d != 0) else np.nan,
                   "RMSE last cycle (µm)": 1e3*gm[gm.Realization == rl]["RMSE last cycle (mm)"].mean(),
                   "Memory torque diff vs L1 (mN m)": gm[gm.Realization == rl]["Memory torque diff vs L1 RMS (mN m)"].mean()}
            sp = s.intersection(pinn.index)
            if len(sp) and model != "oracle (frac)":
                row["PINN RMSE (µm)"] = 1e3*pinn[sp].mean()
                row["gain kept (%)"] = 100*((pinn[sp] - x[sp]).sum()/(pinn[sp] - l1[sp]).sum()) if (pinn[sp] - l1[sp]).sum() > 0 else np.nan
            summ.append(row)
summ = pd.DataFrame(summ); summ.to_csv(f"NB06_fPINN_Summary{SUFFIX}.csv", index=False)
cols = ["Realization", "RMSE mean (µm)", "Δ vs L1 (µm)", "Δ vs L1 (%)", "p (paired t, vs L1)", "gain kept (%)", "RMSE last cycle (µm)", "Memory torque diff vs L1 (mN m)"]
for (alpha, sc, model), g in summ.groupby(["alpha true", "Scenario", "Model"]):
    print(f"\nα = {alpha} | {sc} | {model}" + (f" | PINN {g['PINN RMSE (µm)'].iloc[0]:.1f} µm" if "PINN RMSE (µm)" in g and g["PINN RMSE (µm)"].notna().any() else ""))
    print(g[[c for c in cols if c in g]].round(3).to_string(index=False))

## Experiment 3a: cost per update in Python (NumPy) against the elapsed operation time

The L1 sum is timed with a history of `t/T_C` samples already stored; the other realizations do not depend on t. The control period is 2 ms; a realization that needs a sizeable fraction of it is not usable on the robot.

In [ ]:
def time_push(mem, n_hist, reps):
    mem.reset(max(n_hist, 1) + reps + 2); rng = np.random.default_rng(0)
    if isinstance(mem, MemL1): mem.q_last = np.zeros(3); mem.n = n_hist; mem.dq[1:n_hist + 1] = 1e-4*rng.standard_normal((n_hist, 3))
    else:
        for _ in range(min(n_hist, 2000)): mem.push(1e-2*rng.standard_normal(3))
    qs = 1e-2*rng.standard_normal((reps, 3)); t0 = time.perf_counter()
    for v in qs: mem.push(v)
    return 1e6*(time.perf_counter() - t0)/reps
def flops_bytes(kind, kw, n):
    if kind == "l1":   return 6*n, 8*(4*n)                 # 3 joints x n mult-add; dq history (3n) + weights (n)
    if kind == "win":  Nw = int(round(kw["Tw"]/T_C)); return 6*Nw + (18 if kw.get("tail") else 0), 8*(2*3*Nw + 3*(Nw + 1) + (Nw + 1))
    if kind == "oust": return 3*(4*kw["n"] + 2), 8*(3*kw["n"] + 3*kw["n"] + 6)
    return 0, 0
TIM_METHODS = [r for r in REALIZ if r[1] != "none"]
# One BLAS thread: a controller runs the update on one core, and multithreaded BLAS (MKL on Windows) adds ~100 µs of
# thread start-up per call. The first cell already sets *_NUM_THREADS=1 (works if NumPy was not loaded before);
# this is a second guard: threadpoolctl (old versions crash on some Windows DLLs), then mkl-service; else a warning.
_lim, _mkl_prev = None, None
try:
    from threadpoolctl import threadpool_limits; _lim = threadpool_limits(1); print("BLAS limited to 1 thread (threadpoolctl)")
except Exception as _e:
    try:
        import mkl; _mkl_prev = mkl.set_num_threads_local(1) if hasattr(mkl, "set_num_threads_local") else mkl.get_max_threads(); mkl.set_num_threads(1)
        print(f"threadpoolctl unavailable ({type(_e).__name__}); BLAS limited to 1 thread with mkl-service")
    except Exception:
        print(f"could not limit BLAS threads ({type(_e).__name__}): the L1 times may flatten at ~100 µs (thread start-up). "
              "Fix: pip install -U threadpoolctl, or set OMP_NUM_THREADS=1 before starting Jupyter.")
tim = []
for T in TIMING_T:
    n = int(T/T_C)
    for label, kind, kw in TIM_METHODS:
        if kind != "l1" and T != TIMING_T[0]: us = tim[[x["Realization"] for x in tim].index(label)]["µs per update (Python)"]
        else: us = min(time_push(make_mem(kind, 0.5, **kw), n, 200 if kind == "l1" else 2000) for _ in range(3))
        fl, by = flops_bytes(kind, kw, n)
        tim.append({"Elapsed time (s)": T, "Realization": label, "µs per update (Python)": us, "flops per update": fl, "RAM (kB)": by/1024})
if _lim is not None: _lim.restore_original_limits()
if _mkl_prev is not None: mkl.set_num_threads(_mkl_prev if isinstance(_mkl_prev, int) and _mkl_prev > 0 else os.cpu_count())
tim = pd.DataFrame(tim); tim.to_csv(f"NB06_fPINN_Timing{SUFFIX}.csv", index=False)
print(tim.pivot(index="Realization", columns="Elapsed time (s)", values="µs per update (Python)").round(1).to_string())
print("\nRAM (kB):"); print(tim.pivot(index="Realization", columns="Elapsed time (s)", values="RAM (kB)").round(1).to_string())

## Experiment 3b: the same kernels in plain C99 (for an embedded CPU)

Writes `NB06_fPINN_memkernels.c`: the three update functions (full L1, window + tail, Oustaloup) with static buffers and a benchmark `main` that prints a CSV (ns per update vs history length). If `gcc` is on the PATH it is compiled and run here; otherwise the file is left for an embedded board (Raspberry Pi: `gcc -O2 -std=c99 NB06_fPINN_memkernels.c -o memk -lm && ./memk > NB06_fPINN_TimingC_rpi.csv`). A self-check compares the C output with the Python realizations on the same input.

In [ ]:
C_SRC = r'''
/* NB06_fPINN_memkernels.c - memory term D^alpha q of the fPINN, three realizations (3 joints, double).
   Update functions use no dynamic memory; main() is only a benchmark. Build: gcc -O2 -std=c99 NB06_fPINN_memkernels.c -o memk -lm */
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>
#define NJ 3
static double now_s(void){ struct timespec t; clock_gettime(CLOCK_MONOTONIC, &t); return t.tv_sec + 1e-9*t.tv_nsec; }

/* ---- full-memory L1: D = c * sum_{k=0}^{n-1} b_k dq_{n-k}; b and dq preallocated for n_max ---- */
typedef struct { double c, *b, *dq, ql[NJ]; long n; int started; } L1full;
void l1_update(L1full *s, const double q[NJ], double D[NJ]) {
    int j; long k;
    if (!s->started) { for (j = 0; j < NJ; j++) { s->ql[j] = q[j]; D[j] = 0; } s->started = 1; return; }
    s->n++; for (j = 0; j < NJ; j++) { s->dq[NJ*s->n + j] = q[j] - s->ql[j]; s->ql[j] = q[j]; }
    for (j = 0; j < NJ; j++) { double acc = 0; for (k = 0; k < s->n; k++) acc += s->b[k]*s->dq[NJ*(s->n - k) + j]; D[j] = s->c*acc; }
}
/* ---- window of Nw increments + O(1) tail correction (mean-value form) ---- */
#define NW_MAX 1000
typedef struct { double c, a, bw[NW_MAX + 1], buf[NJ][NW_MAX], qb[NJ][NW_MAX + 1], q0[NJ], ql[NJ], ssum[NJ]; long n; int Nw, started, tail; } Win;
static double l1w(double a, long k){ return pow(k + 1.0, 1 - a) - pow((double)k, 1 - a); }
void win_update(Win *s, const double q[NJ], double D[NJ]) {
    int j, k, Nw = s->Nw;
    if (!s->started) { for (j = 0; j < NJ; j++) { s->q0[j] = s->ql[j] = q[j]; s->qb[j][0] = 0; D[j] = 0; s->ssum[j] = 0; } s->started = 1; return; }
    s->n++; long n = s->n; int i = (int)((n - 1) % Nw); int m = n < Nw ? (int)n : Nw;
    for (j = 0; j < NJ; j++) {
        s->buf[j][i] = q[j] - s->ql[j]; s->ql[j] = q[j];
        double acc = 0; for (k = 0; k < m; k++) { int idx = i - k; if (idx < 0) idx += Nw; acc += s->bw[k]*s->buf[j][idx]; }
        if (s->tail && n > Nw) {
            if (n - Nw - 1 >= 1) s->ssum[j] += s->qb[j][(n - Nw - 1) % (Nw + 1)];   /* read q~_{n-Nw-1} before its slot is overwritten */
            acc += s->bw[Nw]*s->qb[j][(n - Nw) % (Nw + 1)];
            if (n - Nw - 1 >= 1) acc -= (s->bw[Nw] - l1w(s->a, n - 1))*s->ssum[j]/(double)(n - Nw - 1);
        }
        s->qb[j][n % (Nw + 1)] = q[j] - s->q0[j];
        D[j] = s->c*acc;
    }
}
/* ---- Oustaloup, parallel first-order modes (Tustin) ---- */
#define NM_MAX 15
typedef struct { int nm; double K, r[NM_MAX], A[NM_MAX], Bd[NM_MAX], X[NM_MAX][NJ], up[NJ], q0[NJ]; int started; } Oust;
void oust_update(Oust *s, const double q[NJ], double D[NJ]) {
    int j, k; if (!s->started) { for (j = 0; j < NJ; j++) { s->q0[j] = q[j]; s->up[j] = 0; } s->started = 1; }
    for (j = 0; j < NJ; j++) {
        double u = q[j] - s->q0[j], acc = u;
        for (k = 0; k < s->nm; k++) { s->X[k][j] = s->A[k]*s->X[k][j] + s->Bd[k]*(u + s->up[j]); acc += s->r[k]*s->X[k][j]; }
        s->up[j] = u; D[j] = s->K*acc;
    }
}
void oust_init(Oust *s, double a, int nm, double wb, double wh, double h) {
    int N = (nm - 1)/2, i, k; double p[NM_MAX], z[NM_MAX];
    s->nm = nm; s->K = pow(wh, a); s->started = 0;
    for (i = 0; i < nm; i++) { int kk = i - N; p[i] = wb*pow(wh/wb, (kk + N + (1 + a)/2)/(2.0*N + 1)); z[i] = wb*pow(wh/wb, (kk + N + (1 - a)/2)/(2.0*N + 1)); }
    for (i = 0; i < nm; i++) { double num = 1, den = 1; for (k = 0; k < nm; k++) { num *= z[k] - p[i]; if (k != i) den *= p[k] - p[i]; }
        s->r[i] = num/den; s->A[i] = (2 - p[i]*h)/(2 + p[i]*h); s->Bd[i] = h/(2 + p[i]*h); for (k = 0; k < NJ; k++) s->X[i][k] = 0; }
}
int main(int argc, char **argv) {
    double a = 0.5, h = 0.002, D[NJ], q[NJ], sink = 0; long hist[] = {500, 5000, 30000, 300000, 1800000}; int nh = 5, t, j;
    double c = pow(h, -a)/tgamma(2 - a);
    /* self-check input: q_j(t) = sin(t), 0.3 cos(2t), t/6 on 0..6 s; prints D at t = 6 s for the Python check */
    { long n = 3000, i; L1full L; L.c = c; L.n = 0; L.started = 0; L.b = malloc(sizeof(double)*(n + 2)); L.dq = calloc(NJ*(n + 2), sizeof(double));
      Win W = {0}; W.c = c; W.a = a; W.Nw = 250; W.tail = 1; for (i = 0; i <= W.Nw; i++) W.bw[i] = l1w(a, i);
      Oust O; oust_init(&O, a, 7, 1e-2, 1e3, h); double D1[NJ], D2[NJ], D3[NJ];
      for (i = 0; i <= n; i++) L.b[i] = l1w(a, i);
      for (i = 0; i <= n; i++) { double tt = 6.0*i/n; q[0] = sin(tt); q[1] = 0.3*cos(2*tt); q[2] = tt/6; l1_update(&L, q, D1); win_update(&W, q, D2); oust_update(&O, q, D3); }
      printf("check,L1,%.12e,%.12e,%.12e\n", D1[0], D1[1], D1[2]); printf("check,win0.5+tail,%.12e,%.12e,%.12e\n", D2[0], D2[1], D2[2]);
      printf("check,oust7,%.12e,%.12e,%.12e\n", D3[0], D3[1], D3[2]); free(L.b); free(L.dq); }
    printf("method,history_samples,elapsed_s,ns_per_update\n");
    for (t = 0; t < nh; t++) {
        long n = hist[t], i, reps = n > 100000 ? 50 : 500; L1full L; L.c = c; L.started = 1; L.n = n;
        L.b = malloc(sizeof(double)*(n + reps + 2)); L.dq = calloc(NJ*(n + reps + 2), sizeof(double));
        for (i = 0; i < n + reps + 2; i++) L.b[i] = l1w(a, i);
        for (i = 0; i < NJ*(n + 1); i++) L.dq[i] = 1e-4*((i*7919) % 1000 - 500)/500.0;
        for (j = 0; j < NJ; j++) L.ql[j] = 0;
        double t0 = now_s(); for (i = 0; i < reps; i++) { q[0] = 1e-3*i; q[1] = -q[0]; q[2] = 0.5*q[0]; l1_update(&L, q, D); sink += D[0]; }
        printf("L1 full,%ld,%.1f,%.1f\n", n, n*h, 1e9*(now_s() - t0)/reps); free(L.b); free(L.dq);
    }
    { int ws[] = {50, 250, 1000}, w; for (w = 0; w < 3; w++) { Win W = {0}; long i, reps = 200000; W.c = c; W.a = a; W.Nw = ws[w]; W.tail = 1;
        for (i = 0; i <= W.Nw; i++) W.bw[i] = l1w(a, i);
        for (i = 0; i < 5000; i++) { q[0] = 1e-3*sin(1e-3*i); q[1] = q[0]; q[2] = q[0]; win_update(&W, q, D); }
        double t0 = now_s(); for (i = 0; i < reps; i++) { q[0] = 1e-3*sin(1e-3*i); q[1] = q[0]; q[2] = q[0]; win_update(&W, q, D); sink += D[0]; }
        printf("window %g s + tail,%d,-,%.1f\n", W.Nw*h, W.Nw, 1e9*(now_s() - t0)/reps); } }
    { int ms[] = {3, 5, 7, 9, 11}, k; for (k = 0; k < 5; k++) { Oust O; long i, reps = 1000000; oust_init(&O, a, ms[k], 1e-2, 1e3, h);
        double t0 = now_s(); for (i = 0; i < reps; i++) { q[0] = 1e-3*sin(1e-3*i); q[1] = q[0]; q[2] = q[0]; oust_update(&O, q, D); sink += D[0]; }
        printf("Oustaloup %d,%d,-,%.1f\n", ms[k], ms[k], 1e9*(now_s() - t0)/reps); } }
    fprintf(stderr, "sink %g\n", sink); return 0;
}
'''
with open("NB06_fPINN_memkernels.c", "w") as f: f.write(C_SRC)
gcc = shutil.which("gcc")
if gcc is None: print("gcc not found: NB06_fPINN_memkernels.c written; compile it on the target (see the header of this cell).")
else:
    exe = os.path.abspath("NB06_fPINN_memk" + (".exe" if os.name == "nt" else ""))
    r = subprocess.run([gcc, "-O2", "-std=gnu99", "NB06_fPINN_memkernels.c", "-o", exe, "-lm"], capture_output=True, text=True)
    if r.returncode: print("compilation failed:", r.stderr[:2000])
    else:
        out = subprocess.run([exe], capture_output=True, text=True).stdout.strip().splitlines()
        chk = [l.split(",") for l in out if l.startswith("check")]
        _x = np.column_stack([np.sin(np.linspace(0, 6, 3001)), 0.3*np.cos(2*np.linspace(0, 6, 3001)), np.linspace(0, 6, 3001)/6])
        py = {"L1": online(MemL1(0.5), _x)[-1], "win0.5+tail": online(MemWindow(0.5, 0.5, "mean"), _x)[-1], "oust7": online(MemOustaloup(0.5, 7, 1e-2, 1e3), _x)[-1]}
        for c in chk: print(f"C vs Python, {c[1]:12s}: max diff {np.abs(np.array(c[2:], float) - py[c[1]]).max():.1e}")
        from io import StringIO
        timc = pd.read_csv(StringIO("\n".join(l for l in out if not l.startswith("check")))); timc["µs per update"] = timc["ns_per_update"]/1e3
        timc.to_csv(f"NB06_fPINN_TimingC{SUFFIX}.csv", index=False); print(timc.to_string(index=False))

## Figures

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
# (a) offline error vs cost per update (α = 0.5, continuous operation)
g = off[(off.alpha == ALPHA_MAIN) & off.Scenario.str.startswith("Continuous") & (off.Realization != "none")]
for _, r in g.iterrows():
    lab = r.Realization; kind = "oust" if lab.startswith("Oust") else "win"
    kw = dict(n=int(lab.split()[1])) if kind == "oust" else dict(Tw=float(lab.split()[1]), tail="mean" if lab.endswith("tail") else None)
    fl, _ = flops_bytes(kind, kw, 0)
    mk, col = ("o", "tab:blue") if kind == "oust" else ("s", "tab:green") if lab.endswith("tail") else ("^", "0.6") if "lead" in lab else ("v", "tab:orange")
    ax[0].scatter(fl, r["Rel. RMS error (%)"], marker=mk, color=col, s=22)
n60 = int(60/T_C); ax[0].axvline(6*n60, color="k", ls=":", lw=1); ax[0].text(6*n60, 50, " L1 full\n after 60 s", fontsize=7)
ax[0].scatter([], [], marker="o", color="tab:blue", label="Oustaloup"); ax[0].scatter([], [], marker="s", color="tab:green", label="window + tail")
ax[0].scatter([], [], marker="v", color="tab:orange", label="window"); ax[0].scatter([], [], marker="^", color="0.6", label="window + lead term")
ax[0].set_xscale("log"); ax[0].set_yscale("log"); ax[0].set_xlabel("flops per update (3 joints)"); ax[0].set_ylabel("rel. RMS error of D^α q (%)")
ax[0].set_title(f"(a) α = {ALPHA_MAIN}, continuous operation", loc="left", fontsize=9); ax[0].legend(fontsize=7); ax[0].grid(True, which="both", alpha=0.3)
# (b) closed loop: Δ RMSE vs L1 per realization (fPINN, α = 0.5), every scenario
gb = summ[(summ["alpha true"] == ALPHA_MAIN) & (summ.Model.str.startswith("fPINN")) & (~summ.Realization.isin(["L1 full", "none"]))] if len(summ) else summ
for i, sc in enumerate(REFS):
    s = gb[gb.Scenario == sc]
    if len(s): ax[1].barh(np.arange(len(s)) + 0.27*(i - 1), s["Δ vs L1 (%)"], height=0.27, label=sc)
if len(gb):
    labs = gb[gb.Scenario == list(REFS)[0]].Realization.tolist(); ax[1].set_yticks(range(len(labs))); ax[1].set_yticklabels(labs, fontsize=7)
ax[1].axvline(0, color="k", lw=0.8); ax[1].set_xlabel("Cartesian RMSE vs L1 full (%)"); ax[1].set_title("(b) fPINN, CTC + I, mean over seeds ('none' omitted)", loc="left", fontsize=9); ax[1].legend(fontsize=7)
# (c) cost per update vs elapsed operation time
for lab, g in tim.groupby("Realization"):
    if lab == "L1 full" or lab in ("window 0.5 s + tail", "Oustaloup 7 [1e-2, 1e3]"):
        ax[2].plot(g["Elapsed time (s)"], g["µs per update (Python)"], marker="o", label=lab + " (Python)")
if "timc" in globals():
    gc_ = timc[timc.method == "L1 full"]; ax[2].plot(gc_.history_samples*T_C, gc_["µs per update"], marker="s", ls="--", label="L1 full (C)")
ax[2].axhline(1e6*T_C, color="r", ls=":", lw=1); ax[2].text(TIMING_T[0], 1.1e6*T_C, "control period 2 ms", fontsize=7, color="r")
ax[2].set_xscale("log"); ax[2].set_yscale("log"); ax[2].set_xlabel("elapsed operation time (s)"); ax[2].set_ylabel("µs per update")
ax[2].set_title("(c) cost per update on this PC", loc="left", fontsize=9); ax[2].legend(fontsize=7, loc="lower right"); ax[2].grid(True, which="both", alpha=0.3)
fig.tight_layout(); fig.savefig(f"fig_nb06_memoria_finita{SUFFIX}.png", dpi=200); fig.savefig(f"fig_nb06_memoria_finita{SUFFIX}.pdf"); plt.show()